In [ ]:
path_to_dataset = "" #FIXME: Add path to dataset here

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import joblib

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dinov2 = torch.hub.load(
    repo_or_dir="facebookresearch/dinov2", 
    model='dinov2_vits14'
)
dinov2 = dinov2.to(device)
dinov2.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
def extract_features_from_image_crops(csv_path, path_to_dataset):
    """Reads a CSV of bounding boxes, crops images, and extracts DINOv2 features."""
    df = pd.read_csv(csv_path)
    
    features = []
    splits = []
    super_labels = []
    fine_labels = []
    
    for _, row in df.iterrows():
        img_name = row['image_filename']
        subfolder = img_name[0].upper()
        img_path = os.path.join(path_to_dataset, subfolder, img_name)
        
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}. Skipping.")
            continue
            
        try:
            # Load image
            img = Image.open(img_path).convert('RGB')
            img_width, img_height = img.size
            
            # Convert normalized YOLO coordinates to absolute pixel coordinates
            x_center = row['x_center'] * img_width
            y_center = row['y_center'] * img_height
            box_width = row['width'] * img_width
            box_height = row['height'] * img_height
            rotation_angle = row['rotation_angle']
            is_flipped = row['is_flipped']
            
            left = x_center - (box_width / 2)
            top = y_center - (box_height / 2)
            right = x_center + (box_width / 2)
            bottom = y_center + (box_height / 2)
            
            # Crop the image to the bounding box
            crop_img = img.crop((left, top, right, bottom))

            # Rotate the image if necessary based on the 'rotation_angle' column
            if rotation_angle == 90:
                crop_img = crop_img.rotate(-90, expand=True)
            elif rotation_angle == 180:
                crop_img = crop_img.rotate(180, expand=True)
            elif rotation_angle == 270:
                crop_img = crop_img.rotate(90, expand=True)

            # Flip the cropped patch horizontally if flagged
            if is_flipped:
                crop_img = crop_img.transpose(Image.FLIP_LEFT_RIGHT)

            # Prepare tensor and extract features
            img_tensor = transform(crop_img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                feature_vector = dinov2(img_tensor)
                features.append(feature_vector.cpu().numpy().flatten())
                super_labels.append(row['super_category'])
                fine_labels.append(row['fine_category'])
                splits.append(row['split'])
                
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            
    return np.array(features), np.array(super_labels), np.array(fine_labels), np.array(splits)

**Feature extraction**</br>
*Only run these cells if feature vectors could not be downloaded*

In [ ]:
X, super_labels, fine_labels, splits = extract_features_from_image_crops(
    "../Training_data/notch_classification_dataset.csv",
    path_to_dataset
)
print(f"Extracted {len(X)} feature vectors.")

In [ ]:
np.savez_compressed('../assets/dinov2_features.npz', features=X, super_labels=super_labels, fine_labels=fine_labels, splits=splits)

**Load features instead of extracting**

In [ ]:
data = np.load('../assets/dinov2_features.npz', allow_pickle=True) #FIXME: add correct path to dinov2 features

X = data['features']
super_labels = data['super_labels']
fine_labels = data['fine_labels']
splits = data['splits']

print(f"Loaded {len(X)} feature vectors.")

In [ ]:
# Split the dataset into training and validation sets using CSV
train_mask = (splits == 'train')
test_mask = (splits == 'test')

X_train, X_test = X[train_mask], X[test_mask]
y_train_super, y_test_super = super_labels[train_mask], super_labels[test_mask]
y_train_fine, y_test_fine = fine_labels[train_mask], fine_labels[test_mask]

**Training notch-noise SVM**

In [ ]:
notch_noise_classifier = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
notch_noise_classifier.fit(X_train, y_train_super)

# Evaluate the model
y_pred_notch_noise = notch_noise_classifier.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test_super, y_pred_notch_noise))

print("\nClassification Report:")
print(classification_report(y_test_super, y_pred_notch_noise))

In [ ]:
joblib.dump(notch_noise_classifier, '../assets/svm_notch_noise.pkl')

**Training shape SVM with sloped notches combined**

In [ ]:
# Group sloped_left and sloped_right into a single 'sloped' class
y_train_fine_grouped = y_train_fine.copy()
y_train_fine_grouped[y_train_fine_grouped == 'sloped_left'] = 'sloped'
y_train_fine_grouped[y_train_fine_grouped == 'sloped_right'] = 'sloped'

y_test_fine_grouped = y_test_fine.copy()
y_test_fine_grouped[y_test_fine_grouped == 'sloped_left'] = 'sloped'
y_test_fine_grouped[y_test_fine_grouped == 'sloped_right'] = 'sloped'

In [ ]:
svm_shape_grouped = SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42)
svm_shape_grouped.fit(X_train, y_train_fine_grouped)

# Evaluate the model
predicted_as_notch_by_svm1_mask = (y_pred_notch_noise == 'notch')
X_test_passed_svm1 = X_test[predicted_as_notch_by_svm1_mask]
y_test_passed_svm1 = y_test_fine_grouped[predicted_as_notch_by_svm1_mask]

y_pred_shape = svm_shape_grouped.predict(X_test_passed_svm1)

print("Confusion Matrix:")
print(confusion_matrix(y_test_passed_svm1, y_pred_shape))

print("\nClassification Report:")
print(classification_report(y_test_passed_svm1, y_pred_shape))

In [ ]:
joblib.dump(svm_shape_grouped, '../assets/svm_shape_grouped.pkl')

**Training sloped SVM**

In [ ]:
# Filter the ORIGINAL fine labels to get only slopes
slope_train_mask = (y_train_fine == 'sloped_left') | (y_train_fine == 'sloped_right')
slope_test_mask = (y_test_fine == 'sloped_left') | (y_test_fine == 'sloped_right')

X_train_slopes, y_train_dir = X_train[slope_train_mask], y_train_fine[slope_train_mask]
X_test_slopes, y_test_dir = X_test[slope_test_mask], y_test_fine[slope_test_mask]

print(f"Training direction SVM on {len(X_train_slopes)} sloped samples...")

In [ ]:
svm_direction = SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42)
svm_direction.fit(X_train_slopes, y_train_dir)

y_pred = svm_direction.predict(X_test_slopes)

print("Confusion Matrix:")
print(confusion_matrix(y_test_dir, y_pred))

print("\nClassification Report:")
print(classification_report(y_test_dir, y_pred))

In [ ]:
joblib.dump(svm_direction, '../assets/svm_direction.pkl')